# Aula 9 — Dados em Painel: *Mais Armas, Menos Crime?*

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

Os oito capítulos anteriores deixaram uma dívida: o que fazer quando a variável omitida
é **não observável** e não tem boa *proxy*? Dizer "inclua a variável" não ajuda quando
ela é a cultura de um estado ou sua qualidade institucional.

A resposta deste laboratório é de uma economia notável: se não podemos **medir** a
variável omitida, talvez possamos **eliminá-la** — e para isso basta observar cada
unidade mais de uma vez.

> **A ideia central:** se uma variável omitida **não muda ao longo do tempo**, então
> nenhuma **mudança** em $Y$ ao longo do tempo pode ter sido causada por ela.

O caso de teste é o debate **"mais armas, menos crime"** (Lott e Mustard, 1997): leis
de porte de arma do tipo *shall-issue* reduzem a criminalidade? Seis resultados:

1. o MQO agrupado dá um efeito **enorme e negativo** — e é espúrio;
2. efeitos fixos de estado **invertem o sinal** do coeficiente;
3. efeitos fixos de tempo levam o efeito a **praticamente zero**;
4. a transformação *within* é **idêntica** ao LSDV, verificada na máquina;
5. o erro-padrão **clusterizado** é ~2,3× o convencional;
6. um regressor invariante no tempo é **inestimável** — a fronteira do método.

Ao contrário dos laboratórios 1 a 6, este usa **dados reais**: o painel *Guns*,
51 estados americanos observados de 1977 a 1999.


## 0. Preparação

O Colab traz `ggplot2` e `dplyr`; a célula abaixo instala o que faltar. O `plm` é o
pacote canônico de painel em R, e `sandwich`/`lmtest` dão os erros-padrão
clusterizados. O `stargazer` formata as tabelas de regressão.

A instalação leva cerca de um minuto na primeira execução.


In [ ]:
pacotes <- c("plm", "sandwich", "lmtest", "stargazer", "ggplot2", "dplyr")
for (p in pacotes) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}

suppressMessages({
  library(plm); library(sandwich); library(lmtest)
  library(stargazer); library(ggplot2); library(dplyr)
})

theme_set(theme_minimal(base_size = 13) +
  theme(panel.grid.minor = element_blank(),
        legend.position = "bottom", legend.title = element_blank()))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)


## 1. Os dados

O painel *Guns* acompanha 51 unidades (os 50 estados mais o Distrito de Columbia) de
1977 a 1999. Baixamos de uma fonte pública para que o notebook rode no Colab sem
depender de arquivos locais.

| Variável | Descrição |
|:---|:---|
| `vio`, `mur`, `rob` | Taxas de crimes violentos, homicídios e roubos por 100.000 hab. |
| `shall` | **1** se havia lei *shall-issue* em vigor no estado-ano |
| `incarc_rate` | Encarcerados por 100.000 hab. no ano anterior |
| `density` | Habitantes por milha quadrada |
| `avginc` | Renda *per capita*, em milhares de dólares |
| `pm1029` | % da população masculina entre 10 e 29 anos |


In [ ]:
url <- paste0("https://raw.githubusercontent.com/vincentarelbundock/",
              "Rdatasets/master/csv/AER/Guns.csv")
bruto <- read.csv(url)

# harmoniza os nomes com os do arquivo da disciplina (Guns.xls)
guns <- bruto |>
  transmute(
    stateid     = state,
    year        = year,
    vio         = violent,
    mur         = murder,
    rob         = robbery,
    shall       = as.integer(law == "yes"),
    incarc_rate = prisoners,
    density     = density,
    avginc      = income / 1000,
    pop         = population,
    pb1064      = afam,
    pw1064      = cauc,
    pm1029      = male
  ) |>
  mutate(l_vio = log(vio), l_mur = log(mur), l_rob = log(rob))

c(unidades    = length(unique(guns$stateid)),
  periodos    = length(unique(guns$year)),
  observacoes = nrow(guns),
  balanceado  = all(table(guns$stateid) == length(unique(guns$year))))


Painel **balanceado**: $51 \times 23 = 1173$ observações — exatamente o que o Gretl
reporta na tela da aula.

Declaramos a estrutura de painel para o `plm`:


In [ ]:
pguns <- pdata.frame(guns, index = c("stateid", "year"))
pdim(pguns)


## 2. Por que o painel é informativo

Duas figuras antes de qualquer regressão. A primeira mostra o que torna a estimação
possível; a segunda, o que torna os efeitos de tempo necessários.


In [ ]:
adocao <- guns |> group_by(year) |>
  summarise(estados = sum(shall), .groups = "drop")

ggplot(adocao, aes(year, estados)) +
  geom_line(color = az, linewidth = 0.9) +
  geom_point(color = az, size = 1.8) +
  labs(x = NULL, y = "Estados com lei shall-issue")


A adoção é **escalonada no tempo**: de pouquíssimos estados em 1977 a cerca de metade
no fim dos anos 1990. É essa variação *dentro* de cada estado que o estimador de
efeitos fixos explora — sem ela, não haveria o que estimar (seção 7).


In [ ]:
media_ano <- guns |> group_by(year) |>
  summarise(vio = mean(vio), mur = mean(mur), .groups = "drop")

ggplot(media_ano, aes(year)) +
  geom_line(aes(y = vio / max(vio), color = "Crimes violentos"), linewidth = 0.9) +
  geom_line(aes(y = mur / max(mur), color = "Homicídios"), linewidth = 0.9) +
  scale_color_manual(values = c("Crimes violentos" = az, "Homicídios" = vm)) +
  labs(x = NULL, y = "Índice (máximo = 1)")


Aqui está a razão de ser dos **efeitos fixos de tempo**: a criminalidade sobe até o
início dos anos 1990 e despenca em seguida — **em toda parte**, inclusive onde não se
aprovou lei alguma. Quem ignorar essa tendência comum vai atribuir à lei uma queda que
teria ocorrido de todo modo.


### A comparação ingênua

Antes de regredir, a comparação de médias que a tese "mais armas, menos crime" invoca:


In [ ]:
guns |>
  group_by(lei = factor(shall, labels = c("Sem lei", "Com lei"))) |>
  summarise(n = n(), vio = mean(vio), densidade = mean(density),
            renda = mean(avginc), .groups = "drop")


Estados-ano **com** lei têm menos crimes violentos. Mas note a **densidade**: eles são
drasticamente menos densos. Estamos comparando estados rurais do Oeste com estados
urbanos do Nordeste — e a densidade é fixa no estado e correlacionada tanto com o crime
quanto com a propensão a aprovar a lei. É o viés de variável omitida da Aula 5, na sua
forma mais clássica.


## 3. A progressão das especificações

O argumento inteiro do laboratório cabe em quatro regressões.


In [ ]:
# (1) MQO agrupado: ignora a estrutura de painel
m1 <- lm(l_vio ~ shall, data = guns)

# (2) efeitos fixos de estado
m2 <- plm(l_vio ~ shall, data = pguns, model = "within")

# (3) efeitos fixos de estado e de tempo
m3 <- plm(l_vio ~ shall, data = pguns, model = "within", effect = "twoways")

# (4) idem, com controles demograficos
m4 <- plm(l_vio ~ shall + incarc_rate + density + avginc + pop +
            pb1064 + pw1064 + pm1029,
          data = pguns, model = "within", effect = "twoways")

data.frame(
  especificacao = c("MQO agrupado", "EF estado", "EF estado + tempo",
                    "EF duas vias + controles"),
  shall = round(c(coef(m1)[["shall"]], coef(m2)[["shall"]],
                  coef(m3)[["shall"]], coef(m4)[["shall"]]), 4)
)


**A lição do capítulo em quatro números.**

- $-0{,}44$ no MQO agrupado: um efeito espetacular, e é o número que alimentou a tese.
- $+0{,}11$ com efeitos de estado: o coeficiente **troca de sinal**. Toda a associação
  negativa vinha da comparação *entre* estados.
- $\approx 0$ com efeitos de tempo: o que restava era a queda nacional dos anos 1990.
- $\approx 0$ com controles: nada muda, porque os efeitos fixos já fizeram o trabalho.


## 4. A transformação *within*, na mão

O `plm` faz o trabalho, mas vale ver o que ele faz por baixo do capô — é o espírito dos
laboratórios anteriores. A álgebra é a de sempre: centrar cada variável na média da sua
unidade elimina $\alpha_i$, porque $\alpha_i$ é constante no tempo.

$$\tilde{Y}_{it} = Y_{it} - \bar{Y}_i = \beta_1(X_{it} - \bar{X}_i) + (u_{it} - \bar{u}_i)$$


In [ ]:
centrar <- function(x, grupo) x - ave(x, grupo)

guns$lvio_w  <- centrar(guns$l_vio, guns$stateid)
guns$shall_w <- centrar(guns$shall, guns$stateid)

within_mao <- lm(lvio_w ~ shall_w - 1, data = guns)   # sem intercepto
lsdv       <- lm(l_vio ~ shall + factor(stateid), data = guns)

c(plm       = coef(m2)[["shall"]],
  within_mao = coef(within_mao)[["shall_w"]],
  lsdv      = coef(lsdv)[["shall"]])


Os três coincidem até a precisão de máquina. "Estimar efeitos fixos" é literalmente
**centrar os dados e rodar um MQO** — com a diferença de que o LSDV estimou 50
coeficientes de binárias que ninguém vai olhar.

Para as duas dimensões, a centragem subtrai a média da unidade **e** a do período, e
soma de volta a média geral (sem isso, ela seria subtraída duas vezes):


In [ ]:
centrar2 <- function(x, unidade, tempo) {
  x - ave(x, unidade) - ave(x, tempo) + mean(x)
}

guns$lvio_tw  <- centrar2(guns$l_vio, guns$stateid, guns$year)
guns$shall_tw <- centrar2(guns$shall, guns$stateid, guns$year)

duas_vias_mao <- lm(lvio_tw ~ shall_tw - 1, data = guns)

c(plm = coef(m3)[["shall"]], mao = coef(duas_vias_mao)[["shall_tw"]])


## 5. Erros-padrão clusterizados

As observações de um mesmo estado **não são independentes** — é o mesmo estado. A
fórmula robusta usual (HC1) pressupõe independência entre todas as observações e é
**inconsistente** aqui.

O erro-padrão **clusterizado** trata cada unidade como um *cluster*, permitindo
correlação arbitrária dentro dele. Onde o HC1 somava observação a observação, o
clusterizado soma **cluster a cluster**:

$$\widehat{V}_{\text{cluster}} = (X'X)^{-1}\left(\sum_{i=1}^{n} X_i'\hat{u}_i\hat{u}_i'X_i\right)(X'X)^{-1}$$

O termo $\hat{u}_i\hat{u}_i'$ é uma matriz $T \times T$ **cheia** — e é nos elementos
fora da diagonal, que o HC1 zerava, que mora a correlação serial.


In [ ]:
ep_conv <- sqrt(diag(vcov(m4)))[["shall"]]
ep_clus <- sqrt(diag(vcovHC(m4, type = "HC1", cluster = "group")))[["shall"]]

c(convencional = ep_conv, clusterizado = ep_clus, razao = ep_clus / ep_conv)


O clusterizado é cerca de **2,3× maior**. Quem reportasse o convencional trabalharia com
uma estatística $t$ inflada pelo mesmo fator, e correria risco sério de rejeitar
hipóteses nulas verdadeiras.

> Em painel com efeitos fixos, erros-padrão clusterizados por unidade são o **padrão da
> literatura aplicada**. A ausência deles numa tabela é motivo legítimo de crítica em um
> *referee report*.


## 6. A tabela, como ela deve ser apresentada

Reunindo tudo em `stargazer`, com erros-padrão clusterizados em todas as colunas de
painel.


In [ ]:
ep_cluster <- function(m) sqrt(diag(vcovHC(m, type = "HC1", cluster = "group")))

stargazer(m1, m2, m3, m4,
  type = "text",
  se = list(sqrt(diag(vcovHC(m1, type = "HC1"))),
            ep_cluster(m2), ep_cluster(m3), ep_cluster(m4)),
  title = "Leis shall-issue e crimes violentos",
  dep.var.labels = "log(taxa de crimes violentos)",
  column.labels = c("Pooled", "EF estado", "EF 2 vias", "EF 2 vias"),
  covariate.labels = c("shall (lei em vigor)", "Encarceramento", "Densidade",
                       "Renda media", "Populacao", "% negros 10-64",
                       "% brancos 10-64", "% homens 10-29"),
  add.lines = list(
    c("EF de estado", "Nao", "Sim", "Sim", "Sim"),
    c("EF de tempo",  "Nao", "Nao", "Sim", "Sim"),
    c("Controles",    "Nao", "Nao", "Nao", "Sim")),
  omit.stat = c("f", "ser", "adj.rsq"), digits = 4,
  notes = "Erros-padrao clusterizados por estado (col. 1: robustos).",
  notes.align = "l")


## 7. A fronteira do método

O efeito fixo elimina **tudo** que é constante na unidade — inclusive aquilo que
gostaríamos de estimar. Se $X_{it} = X_i$ para todo $t$, então $\bar{X}_i = X_i$ e o
regressor centrado é **identicamente zero**.

Verificando com uma variável fixa por estado:


In [ ]:
guns$dens_fixa <- ave(guns$density, guns$stateid)   # invariante por construcao
guns$dens_w    <- centrar(guns$dens_fixa, guns$stateid)

c(maximo_desvio_absoluto = max(abs(guns$dens_w)))


Zero. Não há o que estimar — a variável foi inteiramente absorvida pelos efeitos fixos.

É a contrapartida exata da virtude do método: **o mesmo mecanismo que elimina a cultura
não observada elimina também a densidade observada**. Qualquer variável invariante no
tempo — gênero, raça, país de nascimento — tem seu coeficiente eliminado.


## 8. Testes de especificação

Duas perguntas que a tabela não responde sozinha.


In [ ]:
# (a) os efeitos de estado sao conjuntamente significantes?
pooled_ctrl <- plm(l_vio ~ shall + incarc_rate + density + avginc + pop +
                     pb1064 + pw1064 + pm1029, data = pguns, model = "pooling")
ef_estado   <- plm(l_vio ~ shall + incarc_rate + density + avginc + pop +
                     pb1064 + pw1064 + pm1029, data = pguns, model = "within")

pFtest(ef_estado, pooled_ctrl)


Rejeita-se com folga: o MQO agrupado é inadequado, como a análise sugeria.


In [ ]:
# (b) efeitos fixos ou aleatorios? Teste de Hausman
ef_aleatorio <- plm(l_vio ~ shall + incarc_rate + density + avginc + pop +
                      pb1064 + pw1064 + pm1029, data = pguns, model = "random")

phtest(ef_estado, ef_aleatorio)


O teste de Hausman compara o estimador de **efeitos fixos** (consistente sempre) com o
de **efeitos aleatórios** (eficiente apenas se $\text{Cov}(\alpha_i, X_{it}) = 0$).

A rejeição indica que o efeito individual **é correlacionado** com os regressores —
exatamente o que se esperava, já que a propensão a aprovar a lei se relaciona com
características permanentes do estado. **Efeitos fixos é a especificação correta.**


## 9. Replicando a saída do Gretl

A tela do Gretl mostrada em aula usa um subconjunto de controles com efeitos fixos de
estado e dummies de tempo (`dt_2` a `dt_23`), erros-padrão agrupados por unidade.


In [ ]:
m_gretl <- plm(l_vio ~ shall + density + avginc + pm1029 + incarc_rate +
                 factor(year), data = pguns, model = "within")

ct <- coeftest(m_gretl, vcov = vcovHC(m_gretl, type = "HC1", cluster = "group"))
round(ct[c("shall", "density", "avginc", "pm1029", "incarc_rate"), ], 7)


Os coeficientes reproduzem a saída do Gretl **dígito a dígito**:

| | Gretl | Aqui |
|:---|---:|---:|
| `shall` | $-0{,}0271992$ | $-0{,}0271992$ |
| `density` | $-0{,}0944334$ | $-0{,}0944334$ |
| `pm1029` | $0{,}0859734$ | $0{,}0859734$ |

O erro-padrão sai $0{,}04166$ contra os $0{,}0420773$ do Gretl. A diferença é o **fator
de correção de amostra finita** do estimador clusterizado — aplicando o mesmo ajuste do
Gretl, $\frac{G}{G-1}\cdot\frac{N-1}{N-K}$:


In [ ]:
G <- length(unique(index(pguns)$stateid))
N <- nobs(m_gretl)
K <- length(coef(m_gretl))
V <- vcovHC(m_gretl, type = "HC0", cluster = "group")

c(gretl     = 0.0420773,
  replicado = sqrt(diag((G/(G-1)) * ((N-1)/(N-K)) * V))[["shall"]])


Coincide até a quinta casa decimal.


## 10. Robustez

A conclusão depende de olharmos crimes violentos?


In [ ]:
ctrl <- "shall + incarc_rate + density + avginc + pop + pb1064 + pw1064 + pm1029"

m_vio <- plm(as.formula(paste("l_vio ~", ctrl)), data = pguns,
             model = "within", effect = "twoways")
m_mur <- plm(as.formula(paste("l_mur ~", ctrl)), data = pguns,
             model = "within", effect = "twoways")
m_rob <- plm(as.formula(paste("l_rob ~", ctrl)), data = pguns,
             model = "within", effect = "twoways")

data.frame(
  dependente = c("log(violentos)", "log(homicidios)", "log(roubos)"),
  shall      = round(c(coef(m_vio)[["shall"]], coef(m_mur)[["shall"]],
                       coef(m_rob)[["shall"]]), 4),
  ep_cluster = round(c(ep_cluster(m_vio)[["shall"]], ep_cluster(m_mur)[["shall"]],
                       ep_cluster(m_rob)[["shall"]]), 4)
) |> mutate(t = round(shall / ep_cluster, 2))


Em nenhuma das três medidas o coeficiente é negativo **e** significante. A conclusão é
robusta à escolha da variável dependente.


## O que levar deste laboratório

**Sobre o método.** Efeitos fixos eliminam variáveis omitidas **constantes no tempo**,
observadas ou não, ao preço de descartar toda a variação entre unidades — e, com ela,
qualquer regressor invariante no tempo (seção 7).

**Sobre o caso.** A evidência a favor de "mais armas, menos crime" **não sobrevive** a
controles elementares: o efeito troca de sinal com efeitos de estado e vai a zero com
efeitos de tempo. É a conclusão de Ayres e Donohue (2003) contra Lott.

**Sobre o que o painel não resolve.** Duas ressalvas honestas:

1. O método nada faz contra variáveis omitidas que **variam dentro do estado** — a
   epidemia de *crack*, mudanças no policiamento, outras leis alteradas simultaneamente.
2. A hipótese de **exogeneidade estrita** exige que não haja *feedback* de $u$ para os
   $X$ futuros. Aqui ela é discutível: legisladores plausivelmente reagem à criminalidade
   ao decidir aprovar a lei. É causalidade simultânea, em versão dinâmica.

O painel move a análise para muito mais perto da identificação causal — mas não a
entrega sozinho.

---

**Referências**

- Ayres, I. e Donohue, J. J. (2003). "Shooting Down the More Guns, Less Crime
  Hypothesis". *Stanford Law Review*, 55(4), 1193–1312.
- Lott, J. R. e Mustard, D. B. (1997). "Crime, Deterrence, and Right-to-Carry Concealed
  Handguns". *Journal of Legal Studies*, 26(1), 1–68.
- Stock, J. H. e Watson, M. W. (2020). *Introduction to Econometrics*, 4ª ed., cap. 10.
